# 📱 DjezzyBot — Colab Runbook (T4 GPU)

A voice + text chatbot for Djezzy, in Arabic / French / English / Darija.

**Do this first:** `Exécution ▸ Modifier le type d'exécution ▸ T4 GPU`.

Then run the steps below from top to bottom. The project lives in your Google
Drive (folder `DjezzyBot_v2`); Step 1 connects to it.

> At startup the bot answers from the cached `data/djezzy_pages.json`. The live
> crawler (the **Rafraîchir** button and the daily 03:00 job) re-scrapes djezzy.dz
> with Playwright/Chromium — Step 1 installs that browser; Step 1b verifies it.

## Step 1 — Setup  (connect Google Drive + install)

> **Re-uploaded the folder into a session that's still running?** Do **Exécution ▸ Redémarrer la session** first, then run top to bottom — otherwise Python keeps the *old* code in memory and your updates won't take effect.

In [ ]:
# Connects the Google Drive folder you uploaded (DjezzyBot_v2), then installs deps.
# A popup will ask you to authorize Google Drive — accept it.
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
PROJ = "/content/drive/MyDrive/DjezzyBot_v2"   # change if you put the folder elsewhere in Drive
assert os.path.isdir(PROJ), f"❌ Not found: {PROJ} — check the folder name/location in your Drive."
os.chdir(PROJ); sys.path.insert(0, PROJ)
print("✅ Project ready in", os.getcwd())

subprocess.run("pip install -q -r requirements.txt", shell=True)

# Playwright's CHROMIUM BINARY is NOT installed by pip — without it the live scraper
# and the daily refresh (scraper.run_scrape / the Refresh button) fail. Install it now.
print("Installing Chromium for the scraper… (one-time, ~150 MB)")
subprocess.run("playwright install chromium", shell=True)
subprocess.run("playwright install-deps chromium", shell=True)  # system libs (Colab is root)

import torch
print("✅ GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "⚠️ NONE — set Exécution ▸ Modifier le type d'exécution ▸ T4 GPU, then re-run")

## Step 1b — Verify the scraper  (optional)
Confirms Chromium is installed and the crawler can render a live djezzy.dz page, so the **Rafraîchir** button and the daily 03:00 refresh will actually work.

In [ ]:
# Step 1b — verify the scraper stack (optional, ~10 s). Renders ONE live page to
# confirm Chromium works, so you know the Refresh button / daily job will run.
import scraper
print(scraper.smoke_test())   # -> {'ok': True, 'url': ..., 'title': ..., 'chars': ...}

# Full refresh (re-crawls the whole site, up to ~45 min, then rebuilds the index):
#   import scheduler
#   print(scheduler.force_refresh())     # -> {'ok': True, 'n_pages': N, ...}
#   store = scheduler.CURRENT_INDEX      # hot-swapped fresh index

## Step 2 — Build the search index  (from the saved data)

In [ ]:
import scraper, indexer
pages = scraper.load_pages()
assert pages, "data/djezzy_pages.json is missing — make sure it came with the project."
store = indexer.build_index(pages)
print(f"✅ Indexed {store.index.ntotal} chunks from {len(pages)} pages.")

## Step 3 — Test it  (optional, takes a few minutes)
Runs the 14 acceptance checks and prints a PASS/FAIL summary.

In [ ]:
import bot, test_scenarios
results = test_scenarios.run_all(store)
test_scenarios._summary(results)

## Step 3b — Robustness (stress) suite  (optional, ~10 min)
25 harder cross-lingual scenarios (budget, out-of-domain, comparison, roaming, named offers…) across Arabic / French / English / Darija. Surfaces weak spots; this is **not** the acceptance contract.

In [ ]:
# Robustness / stress suite: 25 harder, mostly cross-lingual scenarios that probe
# every route in Arabic / French / English / Darija. Failures here are engineering
# findings to investigate, NOT a broken acceptance contract.
import test_robustness
test_robustness._summary(test_robustness.run_all(store))

## Step 3c — Quantitative metrics  (the real numbers for the thesis)
Per-class precision / recall / F1 for language detection and intent routing, retrieval **recall@k vs a no-router baseline** (the router ablation), the **OOD domain gate**, and automatic answer **groundedness**. The LLM sections print live progress.

In [ ]:
# Quantitative metrics: per-class precision/recall/F1 for language & routing,
# retrieval recall@k vs a no-router baseline, and — with the LLM — the OOD domain
# gate + answer groundedness. Sections 4-5 print live per-generation progress
# (~22 generations, ~10-12 min on a T4); the per-line output means it is NOT a hang.
import evaluate
metrics = evaluate.run_all(store, with_llm=True)

## Step 4 — Launch the app
Opens the chatbot (text + voice). Click the public **share** link it prints.

In [ ]:
import app
app.main()